In [ ]:
# imports 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import random
import os
import glob
import numpy as np

# device setup 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # BUT I NEED TO CONNECT TO AWS? 
print("Device:", device)

/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


In [ ]:
from data_processing import build_pair_df, prepare_data

# STEP 1: call data processing functions
pair_df = build_pair_df(
        base_path=".",
        sample_n=None, # all data
        image_dir="images",
        random_state=42
)

train_loader, val_loader, test_loader = prepare_data(
    pair_df,
    seed=42,
    batch_size=64,
)

In [ ]:
# STEP 2: load ResNet architecture
def get_resnet_backbone():
    model = models.resnet50(weights="IMAGENET1K_V1")

    modules = list(model.children())[:-2]  # keep only conv blocks, output = [B, 2048, 7, 7]
    backbone = nn.Sequential(*modules)

    # extract features before the final classifier 
    features = nn.Sequential(*list(backbone.children())[:-1]) 

    # freeze all layers
    for param in backbone.parameters():
        param.requires_grad = False

    # unfreeze last block
    for name, param in backbone.named_parameters():
        if "layer4" in name:    # last block in ResNet50
            param.requires_grad = True

    # unfreeze BatchNorm layers in layer4
    for m in features.modules():
        if isinstance(m, nn.BatchNorm2d):
            for param in m.parameters():
                param.requires_grad = True

    return backbone


In [ ]:
# STEP 3: build RSS-CNN model (two-branch network)
class RSSCNN(nn.Module):
    def __init__(self, backbone, feat_dim=2048): 
        super().__init__() # initialize parent class
        
        self.backbone = backbone # pretrained backbone
        
        # white box: fusion classifier (3 conv layers + 2-unit classifier)
        self.fusion_head = nn.Sequential(
                    nn.Conv2d(feat_dim*2, 512, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Conv2d(512, 512, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Conv2d(512, 512, kernel_size=3, padding=1),
                    nn.ReLU(),
                )
        
        self.fusion_fc = nn.Linear(512 * 7 * 7, 2)
        
        # ranking layers 
        self.ranking = nn.Sequential( 
                    nn.Linear(2048, 4096), # first FC layer
                    nn.ReLU(inplace=True),
                    nn.Linear(4096, 4096), # 2nd FC layer 
                    nn.ReLU(inplace=True),
                    nn.Linear(4096, 1) # 3rd FC layer 1
                    
                )
        
    # define a forward pass 
    def forward_once(self, x):
        # get info for classification 
        conv_feat = self.backbone(x)          # [B, 2048, 7, 7]
        flat_feat = torch.flatten(conv_feat, 1)    # [B, 2048]

        # get info for ranking 
        score = self.ranking(flat_feat)
        return conv_feat, flat_feat, score


    # define running a forward pass on both images 
    def forward(self, imgA, imgB):
        convA, featA, scoreA = self.forward_once(imgA)  # [B, 2048]
        convB, featB, scoreB = self.forward_once(imgB)  # [B, 2048]

        # fusion
        fusion = torch.cat([convA, convB], dim=1)  # [B, 4096]
        fusion_feat = self.fusion_head(fusion)
        fusion_feat = torch.flatten(fusion_feat, 1) 
        logits = self.fusion_fc(fusion_feat)          # [B, 2]

        return logits, scoreA, scoreB

In [ ]:
# STEP 4: loss function

# soft max loss
criterion = nn.CrossEntropyLoss()

def classification_loss(logits, label): 
    return criterion(logits, label)

# ranking loss
def ranking_loss(scoreA, scoreB, y):
    score_diff = scoreB - scoreA
    desired_margin = y * score_diff # we want to maximize this term
    violation_margin = torch.maximum(torch.maximum(torch.tensor(0.0), -desired_margin)) # max(0, -desired_margin)
    squared_loss = violation_margin.pow(2)

    return torch.mean(squared_loss)

# total loss = soft max + ranking loss combined 
def rss_loss(classification_loss, ranking_loss, lamb):
    rss_loss = classification_loss + lamb * ranking_loss 
    return rss_loss 

In [ ]:
# STEP 4b: evaluation loss (no training)
def eval_loss(dataloader, model, loss_fn):
    model.eval()  # Set to evaluation mode
    total_loss = 0

    with torch.no_grad():  # No gradient computation
        for imgA, imgB, label, y in dataloader:
            imgA = imgA.to(device)
            imgB = imgB.to(device)
            label = label.to(device).long()

            logits = model(imgA, imgB)
            loss = loss_fn(logits, label)

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# STEP 5: training loop 
def train_one_epoch(dataloader, model, optimizer, lamb):
    model.train() # set model to training mode
    
    total_loss = 0 # initialize total loss

    for imgA, imgB, label,y in dataloader:
        imgA = imgA.to(device) # move to device
        imgB = imgB.to(device) # move to device
        label = label.to(device).long() # move to device

        logits, scoreA, scoreB = model(imgA, imgB) # forward pass
        

        class_loss = classification_loss(logits, label) # compute classification loss

        y = torch.where(label == 1, 1, -1).float().to(device) 
        rank_loss = ranking_loss(scoreA, scoreB, y) # compute ranking loss

        loss = rss_loss(class_loss, rank_loss, lamb) # calculate rss_loss

        optimizer.zero_grad() # zero gradients
        loss.backward() # backpropagation
        optimizer.step() # update weights

        total_loss += loss.item() # accumulate loss

    return total_loss / len(dataloader) # average loss

In [ ]:
# STEP 6: accuracy 
def accuracy(dataloader, model):
    model.eval() # set model to evaluation mode
    correct = 0 # initialize correct count
    total = 0 # initialize total count

    with torch.no_grad(): # no gradient computation
        for imgA, imgB, label in dataloader: # move to device
            imgA = imgA.to(device) 
            imgB = imgB.to(device)
            label = label.to(device)

            logits, scoreA, scoreB = model(imgA, imgB) # forward pass
            preds = logits.argmax(dim=1) # convert logits to predicted class 

            correct += (preds == label).sum().item() # count correct predictions
            total += label.size(0) # count total samples

    return correct / total


In [ ]:
# STEP 7: accuracy, precision, & recall calculation 

def accuracy_precision_recall(test_dataloader, model):
    model.eval()

    TP = 0  # True Positives
    FP = 0  # False Positives
    FN = 0  # False Negatives
    total_correct = 0
    total = 0

    with torch.no_grad():
        for imgA, imgB, label in test_dataloader:
            imgA = imgA.to(device)
            imgB = imgB.to(device)
            label = label.to(device)  # shape [B]

            logits, _, _ = model(imgA, imgB)
            preds = logits.argmax(dim=1)  # shape [B]

            # accuracy
            total_correct += (preds == label).sum().item()
            total += label.size(0)

            # precision/recall components
            TP += ((preds == 1) & (label == 1)).sum().item()
            FP += ((preds == 1) & (label == 0)).sum().item()
            FN += ((preds == 0) & (label == 1)).sum().item()

    accuracy = total_correct / total
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    return accuracy, precision, recall


In [ ]:
# STEP 8: Training w/ Early Stopping
search_results = []

# Define hyperparameter ranges
lr = .0005
batch_size = 64

# Re-Initialize Model & Optimizer
backbone = get_resnet_backbone()
model = RSSCNN(backbone).to(device)  
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Setup Training Config
num_epochs = 20       
patience = 5          
best_val_acc = 0.0     
patience_counter = 0  
best_model_wts = None  # store best model weights

# Local tracking 
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

# start training epochs 
for epoch in range(num_epochs):
    train_loss = train_one_epoch(train_loader, model, optimizer, lamb=0.15)
    val_loss = eval_loss(val_loader, model, classification_loss)

    train_accuracy = accuracy(train_loader, model)
    val_accuracy = accuracy(val_loader, model)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)
    
    print(f"  Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_accuracy:.4f} | Val Acc: {val_accuracy:.4f}")

    # Early Stopping Logic
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        patience_counter = 0 
        best_model_wts = model.state_dict()  # save best model
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("early stopping now")
            break

# Record Results for this Trial
search_results.append({
    'hyperparameters': {'lr': lr, 'batch_size': batch_size},
    'best_val_acc': best_val_acc,
    'history_acc': val_accuracies,
    'losses': train_accuracies
})


In [ ]:
# STEP 9: Test Set Evaluation
# load best model weights
model.load_state_dict(best_model_wts)

# calculate test metrics
test_accuracy, test_precision, test_recall = accuracy_precision_recall(test_loader, model)

# calculate true negatives for completeness
TN = 0
TP_test = 0
FP_test = 0
FN_test = 0

model.eval()
with torch.no_grad():
    for imgA, imgB, label, y in test_loader:
        imgA = imgA.to(device)
        imgB = imgB.to(device)
        label = label.to(device)

        logits, scoreA, scoreB = model(imgA, imgB)
        preds = logits.argmax(dim=1)

        TP_test += ((preds == 1) & (label == 1)).sum().item()
        FP_test += ((preds == 1) & (label == 0)).sum().item()
        FN_test += ((preds == 0) & (label == 1)).sum().item()
        TN += ((preds == 0) & (label == 0)).sum().item()

# print test results
print(f"\nTest Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Positives (TP):  {TP_test}")
print(f"  False Positives (FP): {FP_test}")
print(f"  False Negatives (FN): {FN_test}")
print(f"  True Negatives (TN):  {TN}")
print(f"\nTest Set Size: {TP_test + FP_test + FN_test + TN}")
print("="*50)

In [ ]:
# Adding random search for hyperparameter tuning
import random

# Define hyperparameter ranges
learning_rates = [1e-4, 5e-4, 1e-3, 5e-3]
batch_sizes = [32, 64, 128]
lambdas = [0.1, 0.15, 0.2, 0.5, 1.0]
num_trials = 5  # Number of random search trials


search_results = []

for trial in range(num_trials):
    lr = random.choice(learning_rates)
    batch_size = random.choice(batch_sizes)
    lamb = random.choice(lambdas)
    
    print(f"\n=== Trial {trial+1} | lr={lr}, batch_size={batch_size}, lambda={lamb} ===")
    
    train_losses = []
    val_accuracies = []
    num_epochs = 10 
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(train_loader, model, optimizer, lamb)
        val_acc = accuracy(val_loader, model)
        train_losses.append(train_loss)
        val_accuracies.append(val_acc)
        print(f"Epoch {epoch+1}/{num_epochs}  Loss: {train_loss:.4f}  Acc: {val_acc:.4f}")
    
    # Track results for each trial
    search_results.append({
        'trial': trial+1,
        'lr': lr,
        'batch_size': batch_size,
        'lambda': lamb,
        'train_losses': train_losses,
        'val_accuracies': val_accuracies
    })

# Optionally, print summary of best trial
best_trial = max(search_results, key=lambda x: max(x['val_accuracies']))
print("\nBest trial:", best_trial)

# convert search results to csv
pd.DataFrame(search_results).to_csv("random_search_results.csv", index=False)